# 1. Avaliação independente do experimento
Selecione um experimento concluído. A leitura não cria treinamento e não altera seus artefatos.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from basket_price_prediction.config import ROOT, DATASET, SPLITS, FEATURES
from basket_price_prediction.data import load_data
from basket_price_prediction.preprocessing import prepare_monthly
from basket_price_prediction.features import build_supervised, temporal_split
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
import json
from IPython.display import Image
from basket_price_prediction.experiments import load_experiment, file_hash
from basket_price_prediction.evaluation import group_performance, calculate_metrics
from basket_price_prediction.config import RESULTS
completed=[p for p in RESULTS.iterdir() if p.is_dir() and (p/"manifest.json").exists() and json.loads((p/"manifest.json").read_text()).get("status")=="complete"]
if not completed: raise RuntimeError("Nenhum experimento concluído. Execute o treinamento no notebook 2.")
EXPERIMENT = sorted(completed)[-1]  # Substitua pelo caminho de uma execução para selecioná-la explicitamente.
exp=load_experiment(EXPERIMENT); test=exp["test"]; threshold=exp["config"]["selection"]["threshold"]
print("Experimento:",EXPERIMENT.name, "limiar:",threshold)

## 2. Verificação da fonte, separação e modelo salvo
O hash verifica o arquivo de origem. A igualdade das probabilidades reconstruídas verifica o pipeline e as mesmas observações do teste.

In [ ]:
assert file_hash(DATASET)==exp["manifest"]["dataset_sha256"], "A fonte mudou desde o treinamento"
monthly,_=prepare_monthly(load_data()); supervised,_=build_supervised(monthly); parts=temporal_split(supervised,SPLITS)
assert len(test)==len(parts["test"])
reloaded_probability=exp["model"].predict_proba(parts["test"][FEATURES])[:,1]
assert np.allclose(reloaded_probability,test.probability,rtol=1e-10,atol=1e-10)
assert test.target.tolist()==parts["test"].target.tolist()
print("Hash, registros, rótulos e probabilidades do modelo salvo conferem.")
display(pd.DataFrame(exp["summary"]["splits"]).T)

## 3. Métricas e referências
A precisão indica quantos alertas corresponderam a altas reais. O recall indica quantas altas reais receberam alerta. O F1 combina as duas medidas. Nunca alertar favorece a acurácia quando predominam observações sem alta. Sempre alertar captura todas as altas e também sinaliza todos os casos sem alta. Repetir a alta anterior usa apenas a última variação da série. As três regras usam as mesmas observações do teste do modelo.

In [ ]:
metrics = exp["metrics"].copy()
if "reference_test_always_one" not in metrics:
    # Compatibilidade com execuções anteriores à inclusão da referência.
    metrics["reference_test_always_one"] = calculate_metrics(test.target, np.ones(len(test)), 0.5)
methods = {
    "test": "Regressão logística",
    "reference_test_always_zero": "Nunca emitir alerta",
    "reference_test_always_one": "Sempre emitir alerta",
    "reference_test_previous_increase": "Repetir a alta anterior",
}
comparison = pd.DataFrame({name: metrics[key] for key, name in methods.items()}).T
comparison["falsos alertas"] = comparison.confusion_matrix.map(lambda cm: cm[0][1])
comparison["altas identificadas"] = comparison.confusion_matrix.map(lambda cm: cm[1][1])
display(comparison[["n", "positives", "altas identificadas", "falsos alertas",
                    "precision", "recall", "f1", "accuracy"]])
display(pd.DataFrame(metrics).T)
print((EXPERIMENT/"reports"/"classification_report.txt").read_text())

## 4. Matriz de confusão, curvas e probabilidades
A curva precisão–recall deve ser comparada com a prevalência. ROC-AUC mede ordenação; Brier avalia probabilidades. A calibração em decis é descritiva e não garante calibração futura.

In [ ]:
for name in ["confusion_matrix.png","precision_recall_curve.png","roc_curve.png","probability_distribution.png","calibration_curve.png","threshold_tradeoff.png"]:
    display(Image(filename=str(EXPERIMENT/"figures"/name)))

## 5. Desempenho temporal
Para cada mês-alvo são apresentados tamanho, positivos, precisão, recall, F1 e taxa prevista de positivos. Grupos sem positivos têm recall e F1 indefinidos. A ausência de data de coleta limita a confirmação da disponibilidade real da informação.

In [ ]:
months=group_performance(test,"target_month",threshold).sort_values("target_month")
display(months[["target_month","n","positives","precision","recall","f1","positive_prediction_rate"]]); display(Image(filename=str(EXPERIMENT/"figures"/"monthly_performance.png")))
observed_figure = EXPERIMENT / "figures" / "monthly_observed_increases.png"
if observed_figure.exists():
    display(Image(filename=str(observed_figure)))

## 6. Produtos e supermercados
Cada tabela mostra o total de exemplos, as altas reais e os erros. Os gráficos destacam grupos com pelo menos 30 exemplos e recall definido, com as contagens ao lado dos rótulos nas execuções novas. O número de altas reais importa porque é o denominador do recall. As diferenças entre grupos são descritivas.

In [ ]:
for col, label in [("product", "Produto"), ("supermarket", "Supermercado")]:
    groups = group_performance(test, col, threshold)
    columns = {col: label, "n": "Exemplos", "positives": "Altas reais",
               "precision": "Precisão", "recall": "Recall", "f1": "F1",
               "false_positives": "Falsos alertas", "false_negatives": "Altas sem alerta"}
    display(groups[list(columns)].sort_values("n", ascending=False).rename(columns=columns))
    display(Image(filename=str(EXPERIMENT/"figures"/f"{col}_performance.png")))

## 7. Análise de erros
Exemplos com identificadores, mês de previsão e alvo, variação observada, probabilidade e classes. Uma previsão positiva equivocada é falso alarme; uma negativa equivocada deixa de antecipar a alta.

In [ ]:
for name in ["true_positive","true_negative","false_positive","false_negative"]:
    print(name); display(pd.read_csv(EXPERIMENT/"reports"/f"{name}_examples.csv").head(5))

## 8. Interpretação dos coeficientes
Coeficientes são associações condicionais no logito após padronização/codificação; a magnitude depende da escala, frequência e regularização, e não implica causalidade. Categorias raras e atributos de preço correlacionados dificultam interpretação.

In [ ]:
display(Image(filename=str(EXPERIMENT/"figures"/"coefficients.png")))
model=exp["model"]; coeff=pd.DataFrame({"atributo":model.named_steps["preprocess"].get_feature_names_out(),"coeficiente":model.named_steps["classifier"].coef_[0]})
display(coeff.assign(magnitude=coeff.coeficiente.abs()).sort_values("magnitude",ascending=False).head(20))

## 9. Limitações e resposta à questão
A amostra condiciona-se à continuidade de três meses de histórico e disponibilidade do alvo, podendo selecionar séries mais estáveis. Marcas ausentes podem misturar itens; unidades e descrições têm ambiguidades. A coleta posterior a outubro/2025 não tem data individual registrada, o que impede verificar sua disponibilidade efetiva. Inflação, custos, promoções e mudanças de coleta não estão explicitamente representados. Comparações por grupo são descritivas; resultados não garantem movimentos futuros.

In [ ]:
m = metrics["test"]
always = metrics["reference_test_always_one"]
never = metrics["reference_test_always_zero"]
previous = metrics["reference_test_previous_increase"]
cm = m["confusion_matrix"]
display(Markdown(
    f"A regressão logística identificou **{cm[1][1]} de {m['positives']} altas** "
    f"(recall **{m['recall']:.1%}**) e emitiu **{cm[0][1]} falsos alertas** "
    f"(precisão **{m['precision']:.1%}**). Sempre alertar identificou todas as altas, "
    f"com **{always['confusion_matrix'][0][1]} falsos alertas**. Nunca alertar teve "
    f"acurácia de **{never['accuracy']:.1%}** e deixou passar todas as altas. "
    f"O F1 do modelo foi **{m['f1']:.3f}**, ante **{always['f1']:.3f}** para sempre "
    f"alertar e **{previous['f1']:.3f}** para repetir a alta anterior. O teste já foi "
    "analisado e o limiar de alerta continua sendo aquele selecionado na validação."
))